# 03 · Preparación de los Datos
### Fase 2 y 4 (parcial) de CRISP-DM — OE1

Este notebook sigue una estructura secuencial de 7 etapas, para que el flujo sea fácil de
seguir y de citar en el capítulo metodológico de la tesis:

1. **Carga** de la base cruda.
2. **Calidad** — diagnóstico de nulos, duplicados y consistencia de la granularidad.
3. **Limpieza** — filtro temporal y tratamiento de nulos.
4. **Transformaciones** — `Franja_Horaria`, tipos de datos.
5. **Ingeniería de variables** — grid completo y conteo de accidentes.
6. **Construcción del dataset final** — variable objetivo `Alto_Riesgo` y validación previa a exportar.
7. **Exportación** — guardado en `data/processed/`.

> Nota: `Vehiculos` y `Actor_vial` no se integran en este notebook porque tienen granularidad distinta (por vehículo / por persona involucrada). Se integrarán más adelante como variables adicionales agregadas por `Codigo_Accidente`, si el EDA lo justifica.


In [32]:
import pandas as pd
import numpy as np
import itertools
from pathlib import Path

pd.set_option('display.max_columns', 50)

RAW_PATH = "../data/raw/base-anuario-de-siniestralidad-2024.xlsx"
PROCESSED_PATH = Path('../data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)


## 1. Carga

Se cargan todas las hojas del archivo en un solo paso, separadas en variables independientes.
Esto permite explorar y validar cada hoja por separado (distinta granularidad cada una) antes
de decidir cuáles se integran al dataset de este notebook.


In [33]:
hojas = pd.read_excel(RAW_PATH, sheet_name=None)

siniestros = hojas['Siniestros']
vehiculos = hojas['Vehiculos']
Actor_vial = hojas['Actor_vial']
Diccionario = hojas['Diccionario']

for nombre, hoja in hojas.items():
    print(f'{nombre:15s} -> {hoja.shape[0]:>8,} filas | {hoja.shape[1]:>3} columnas')


Siniestros      ->  278,614 filas |  40 columnas
Vehiculos       ->  523,168 filas |  32 columnas
Actor_vial      ->  608,155 filas |  44 columnas
Diccionario     ->      110 filas |   3 columnas


### Interpretación

El archivo trae 4 hojas de granularidad distinta:

- **`Siniestros`**: un registro por accidente (clave `Codigo_Accidente`) — es la hoja base de este notebook.
- **`Vehiculos`**: un registro por vehículo involucrado en cada accidente (varios por `Codigo_Accidente`).
- **`Actor_vial`**: un registro por persona involucrada en cada accidente.
- **`Diccionario`**: no es data operativa, es la documentación de las columnas del archivo.

`Vehiculos` y `Actor_vial` no se transforman en este notebook, pero quedan cargadas y disponibles
para explorarlas por separado o para una futura agregación por `Codigo_Accidente` si el EDA lo justifica.


In [34]:
siniestros.head()


,Codigo_Accidente,Formulario,Longitud,Latitud,Direccion,Fecha_Acc,AA_Acc,MM_Acc,DD_Mes_Acc,Dia_Semana_Acc,Hora_Acc,Min_Acc,Localidad,Clase_Acc,Elemento_Choque,Tipo_Objeto_Fijo,Gravedad_Indicador_Tradicional,Gravedad_indicador_30d,Con_Bicicleta,Con_Carga,Con_Embriaguez,Con_Huecos,Con_Menores,Con_Moto,Con_Peaton,Con_Persona_Mayor,Con_Rutas,Con_Tpi,Con_Tpp,Con_Velocidad,Con_Sitp,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Provisional,Con_Articulado,Con_Biarticulado,Con_Padron_Dual,Con_Servicio_Especial,Con_Taxi
0,4404210,A685,-74.181387,4.621978,AV AVENIDA CIUDAD DE CALI-KR 55 S 02,2015-02-09,2015,Febrero,9,lunes,20,40,BOSA,Choque,Vehículo,NaN,Con Muertos,Con Muertos,SI,NaN,NaN,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4404211,A666,-74.027304,4.762901,AK 7-CL 186 02,2015-02-06,2015,Febrero,6,viernes,15,50,USAQUÉN,Choque,Vehículo,NaN,Solo Daños,Solo Daños,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN
2,4404212,A698,-74.053529,4.706939,KR 45-CL 127 02,2015-02-07,2015,Febrero,7,sábado,12,30,USAQUÉN,Choque,Vehículo,NaN,Solo Daños,Solo Daños,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4404213,A000042495,-74.099563,4.576256,KR 12-CL 24 S 06,2015-02-09,2015,Febrero,9,lunes,13,45,RAFAEL URIBE URIBE,Volcamiento,NaN,NaN,Con Muertos,Con Muertos,SI,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4404214,A644,-74.091667,4.524080,CL 70-KR 11D SE 04,2015-02-08,2015,Febrero,8,domingo,10,58,SAN CRISTÓBAL,Choque,Vehículo,NaN,Solo Daños,Solo Daños,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
vehiculos.head()


,Codigo_Accidente,Formulario,Fecha_Acc,AA_Acc,Codigo_Vehiculo,Clase,Servicio,Modalidad,Vehiculo_Viajaba_Clasificado,Tipo_SITP,Con_Bicicleta,Con_Carga,Con_Embriaguez,Con_Huecos,Con_Menores,Con_Moto,Con_Peaton,Con_Persona_Mayor,Con_Rutas,Con_Tpi,Con_Tpp,Con_Velocidad,Con_Sitp,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Provisional,Con_Articulado,Con_Biarticulado,Con_Padron_Dual,Con_Servicio_Especial,Con_Taxi
0,4401718,A000040450,2015-01-07,2015,2.0,Motocicleta,Particular,NaN,MOTOCICLETA,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4401718,A000040450,2015-01-07,2015,3.0,NaN,Sin información,NaN,SIN INFORMACIÓN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4401726,A000040954,2015-01-07,2015,1.0,Bus,Público,NaN,TRANSPORTE DE PASAJEROS,BIARTICULADO,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN,SI,NaN,SI,SI,NaN,NaN,NaN,NaN,SI,NaN,NaN,NaN
3,4401725,A000040171,2015-01-07,2015,1.0,Automovil,Particular,NaN,LIVIANO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN
4,4401725,A000040171,2015-01-07,2015,2.0,Bus,Público,NaN,TRANSPORTE DE PASAJEROS,PADRON DUAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,SI,SI,NaN,NaN,NaN,NaN,NaN,SI,NaN,NaN


In [36]:
Actor_vial.head()


,Codigo_Accidentado,Codigo_Accidente,Formulario,Codigo_Vehiculo,Ccodigo_Victima,FechaAcc,AnnoAcc,mesAcc,DD_Mes_Acc,Dia_Semana_Acc,hourAcc,minuAcc,Localidad,Edad,Sexo,Gravedad_Indicador_Tradicional,Muerte_Posterior,Fecha_CambioGravedad,Gravedad_Indicador_30d,Condicion,Condicion_Especifica,Tipo_SITP,Con_Bicicleta,Con_Carga,Con_Embriaguez,Con_Huecos,Con_Menores,Con_Moto,Con_Peaton,Con_Persona_Mayor,Con_Rutas,Con_Tpi,Con_Tpp,Con_Velocidad,Con_Sitp,Con_Troncal,Con_Alimentador,Con_Zonal,Con_Provisional,Con_Articulado,Con_Biarticulado,Con_Padron_Dual,Con_Servicio_Especial,Con_Taxi
0,2456665,4403353,A000041152,2.0,0,2015-01-28,2015,Enero,28,miércoles,16,35,Bosa,20.0,MASCULINO,ILESO,NaN,NaT,ILESO,CONDUCTOR,Cond TPP,ZONAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,SI,NaN,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,SI
1,2456666,4403354,A000042570,1.0,0,2015-01-28,2015,Enero,28,miércoles,7,30,Suba,42.0,MASCULINO,ILESO,NaN,NaT,ILESO,CONDUCTOR,Cond Livianos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2456667,4403354,A000042570,2.0,0,2015-01-28,2015,Enero,28,miércoles,7,30,Suba,43.0,MASCULINO,ILESO,NaN,NaT,ILESO,CONDUCTOR,Cond Livianos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2456670,4403356,A000042404,1.0,0,2015-01-28,2015,Enero,28,miércoles,9,0,Usaquén,51.0,MASCULINO,ILESO,NaN,NaT,ILESO,CONDUCTOR,Cond TPP,ZONAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN
4,2456671,4403356,A000042404,2.0,0,2015-01-28,2015,Enero,28,miércoles,9,0,Usaquén,46.0,MASCULINO,ILESO,NaN,NaT,ILESO,CONDUCTOR,Cond Livianos,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SI,NaN,SI,NaN,NaN,SI,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
sin = siniestros.copy()


## 2. Calidad

Antes de transformar nada, se diagnostica la calidad de la base: unicidad de la clave primaria,
completitud de las variables críticas para el análisis, y consistencia de la ventana temporal.


In [38]:
assert sin['Codigo_Accidente'].is_unique, "Codigo_Accidente tiene duplicados — revisar antes de continuar"
print('OK: Codigo_Accidente es una clave única.', sin['Codigo_Accidente'].nunique(), 'accidentes.')


OK: Codigo_Accidente es una clave única. 278614 accidentes.


In [39]:
print('Años disponibles en el archivo:', sorted(sin['AA_Acc'].unique()))
print()
print('Nulos en variables clave (Localidad, Hora_Acc, Fecha_Acc):')
print(sin[['Localidad', 'Hora_Acc', 'Fecha_Acc']].isnull().sum())


Años disponibles en el archivo: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Nulos en variables clave (Localidad, Hora_Acc, Fecha_Acc):
Localidad    0
Hora_Acc     0
Fecha_Acc    0
dtype: int64


### Interpretación

`Codigo_Accidente` es una clave primaria válida (una fila = un accidente), sin duplicados.
Las variables `Localidad`, `Hora_Acc` y `Fecha_Acc` — necesarias para la granularidad
Localidad × Franja_Horaria del proyecto — no presentan valores nulos. El archivo trae registros
desde 2015, tres años por fuera del alcance definido en el anteproyecto (2018–2024), lo que se
corrige en la sección de Limpieza.


## 3. Limpieza

### 3.1 Filtro de ventana temporal (2018–2024)

El anteproyecto delimita el estudio al periodo 2018–2024. Se filtra explícitamente para no
arrastrar años fuera del alcance de investigación.


In [40]:
sin = sin[sin['AA_Acc'].between(2018, 2024)].copy()
print('Filas tras filtro 2018-2024:', len(sin))


Filas tras filtro 2018-2024: 177099


### Interpretación

Se eliminaron los registros correspondientes a los años 2015–2017, ya que el alcance del
proyecto comprende únicamente el periodo 2018–2024. Esto garantiza la coherencia entre los
datos utilizados y los objetivos de investigación.


### 3.2 Tratamiento de valores nulos — columnas `Con_*`

Confirmado con `value_counts(dropna=False)`: estas columnas son flags binarios donde el nulo significa
**"no aplica"**, no un dato faltante real (ej. `Con_Bicicleta` solo tiene `"SI"` o `NaN`, nunca `"NO"` explícito).


In [41]:
con_cols = [c for c in sin.columns if c.startswith('Con_')]
print(f'{len(con_cols)} columnas Con_* detectadas:')
print(con_cols)


22 columnas Con_* detectadas:
['Con_Bicicleta', 'Con_Carga', 'Con_Embriaguez', 'Con_Huecos', 'Con_Menores', 'Con_Moto', 'Con_Peaton', 'Con_Persona_Mayor', 'Con_Rutas', 'Con_Tpi', 'Con_Tpp', 'Con_Velocidad', 'Con_Sitp', 'Con_Troncal', 'Con_Alimentador', 'Con_Zonal', 'Con_Provisional', 'Con_Articulado', 'Con_Biarticulado', 'Con_Padron_Dual', 'Con_Servicio_Especial', 'Con_Taxi']


In [42]:
# Confirmación del patrón antes de imputar (ej. Con_Bicicleta)
print(sin['Con_Bicicleta'].value_counts(dropna=False))


Con_Bicicleta
NaN    160383
SI      16716
Name: count, dtype: int64


In [43]:
sin[con_cols] = sin[con_cols].fillna('NO')

# Verificación: ya no deberían quedar nulos en estas columnas
print('Nulos restantes en columnas Con_*:', sin[con_cols].isnull().sum().sum())


Nulos restantes en columnas Con_*: 0


### Interpretación

Los valores nulos en las columnas `Con_*` no representan información faltante, sino la ausencia
del atributo correspondiente en ese accidente (por ejemplo, un accidente sin bicicleta involucrada
simplemente no registra el flag). Se imputó `"NO"` de forma explícita para dejar la codificación
binaria completa y evitar que un tratamiento estadístico de nulos (imputación por media, eliminación
de filas) distorsione el significado real del dato.


## 4. Transformaciones

### `Franja_Horaria`

La variable `Franja_Horaria` se creó con el propósito de reducir la variabilidad de la hora exacta
del accidente y facilitar el análisis temporal mediante categorías homogéneas de 6 horas cada una.


In [44]:
def asignar_franja(hora):
    if 0 <= hora < 6:
        return 'Madrugada'
    elif 6 <= hora < 12:
        return 'Mañana'
    elif 12 <= hora < 18:
        return 'Tarde'
    else:
        return 'Noche'

sin['Franja_Horaria'] = sin['Hora_Acc'].apply(asignar_franja)
sin['Fecha_Acc'] = pd.to_datetime(sin['Fecha_Acc']).dt.date

sin['Franja_Horaria'].value_counts()


Franja_Horaria
Tarde        62085
Mañana       55742
Noche        43505
Madrugada    15767
Name: count, dtype: int64

### Interpretación

La franja horaria **Tarde (12:00–17:59)** concentra la mayor cantidad de accidentes registrados, con **62.085 casos**, equivalente aproximadamente al **35,1 %** del total analizado. Le siguen las franjas **Mañana**, con 55.742 accidentes (31,5 %), **Noche**, con 43.505 (24,6 %), y **Madrugada**, con 15.767 (8,9 %).

Estos resultados muestran que la distribución de los accidentes no es uniforme entre las franjas horarias, presentándose una mayor concentración durante la tarde y la mañana.

La conversión de `Fecha_Acc` al tipo `date`, eliminando el componente horario, permite establecer una granularidad diaria y prepara la variable para la posterior agregación por localidad, franja horaria y fecha.

## 5. Ingeniería de variables

**Decisión de diseño:**

Realizar un agregado directo por `Localidad × Franja_Horaria` produciría únicamente **80 combinaciones** (20 localidades × 4 franjas horarias), lo que eliminaría la dimensión temporal del análisis. Por esta razón, se construye un **grid diario**, donde cada fila representa una combinación única de `Localidad`, `Franja_Horaria` y `Fecha_Acc`.

El grid incluye tanto los días en los que se registraron accidentes como aquellos en los que **no se registró ningún accidente** (`Num_Accidentes = 0`). De esta manera, se conserva la información correspondiente a los periodos sin siniestros y se evita trabajar únicamente con las observaciones en las que ocurrió al menos un accidente.

Esta estructura permite representar de forma completa la variación diaria de la siniestralidad y proporciona una base adecuada para la posterior construcción de la variable objetivo `Alto_Riesgo`.

In [45]:
# Conteo real de accidentes por combinación Localidad-Franja-Fecha
conteo = (
    sin.groupby(['Localidad', 'Franja_Horaria', 'Fecha_Acc'])
    .size()
    .reset_index(name='Num_Accidentes')
)
print('Combinaciones con al menos 1 accidente:', len(conteo))


Combinaciones con al menos 1 accidente: 94234


In [46]:
# Grid completo: todas las combinaciones posibles, incluyendo días sin accidentes
localidades = sin['Localidad'].unique()
franjas = ['Madrugada', 'Mañana', 'Tarde', 'Noche']
fechas = pd.date_range('2018-01-01', '2024-12-31', freq='D').date

grid = pd.DataFrame(
    list(itertools.product(localidades, franjas, fechas)),
    columns=['Localidad', 'Franja_Horaria', 'Fecha_Acc']
)
print('Filas del grid completo:', len(grid))

dataset = grid.merge(conteo, on=['Localidad', 'Franja_Horaria', 'Fecha_Acc'], how='left')
dataset['Num_Accidentes'] = dataset['Num_Accidentes'].fillna(0).astype(int)

dataset['Num_Accidentes'].describe()


Filas del grid completo: 204560


count    204560.000000
mean          0.865756
std           1.278405
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          14.000000
Name: Num_Accidentes, dtype: float64

### Interpretación

El dataset resultante contiene **204.560 filas**, correspondientes a las combinaciones de **20 localidades × 4 franjas horarias × 2.557 días** comprendidos entre 2018 y 2024. Esta estructura permite conservar la dimensión temporal del fenómeno, en lugar de realizar únicamente un agregado por localidad y franja horaria, que produciría solo 80 combinaciones.

El **54 % de las combinaciones diarias no registró ningún accidente (`Num_Accidentes = 0`)**. Estos registros son relevantes para el análisis, ya que representan explícitamente los días y franjas en los que no se presentaron accidentes y evitan que el dataset considere únicamente los casos en los que ocurrió algún siniestro.

La inclusión de estas observaciones permite construir una representación más completa de la frecuencia de accidentes y proporciona una base adecuada para la posterior definición de la variable `Alto_Riesgo`.

### 5.1 Variables predictoras — calendario

`Fecha_Acc` en su forma cruda no generaliza (una fecha específica no se repite). Se derivan
variables de calendario, que sí son válidas como predictoras porque se conocen de antemano
para cualquier fecha futura — no dependen de si hubo o no accidentes ese día.


In [47]:
%pip install holidays


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [48]:
import holidays

festivos_co = holidays.Colombia(years=range(2018, 2025))

dataset['Fecha_Acc'] = pd.to_datetime(dataset['Fecha_Acc'])

dataset['Dia_Semana'] = dataset['Fecha_Acc'].dt.day_name()
dataset['Mes'] = dataset['Fecha_Acc'].dt.month
dataset['Es_Fin_de_Semana'] = dataset['Fecha_Acc'].dt.dayofweek.isin([5, 6]).astype(int)
dataset['Es_Festivo'] = dataset['Fecha_Acc'].dt.date.astype('object').isin(festivos_co).astype(int)

dataset[['Fecha_Acc', 'Dia_Semana', 'Mes', 'Es_Fin_de_Semana', 'Es_Festivo']].head()


,Fecha_Acc,Dia_Semana,Mes,Es_Fin_de_Semana,Es_Festivo
0,2018-01-01,Monday,1,0,1
1,2018-01-02,Tuesday,1,0,0
2,2018-01-03,Wednesday,1,0,0
3,2018-01-04,Thursday,1,0,0
4,2018-01-05,Friday,1,0,0


### Interpretación

`Es_Festivo` es especialmente relevante para capturar días atípicos de movilidad. En cuanto al
hallazgo del anteproyecto de que los **viernes y sábados** concentran la mayor proporción de
víctimas, esa señal la captura la variable `Dia_Semana` (que distingue los 7 días individualmente),
no `Es_Fin_de_Semana` — esta última solo marca sábado y domingo como 1, por lo que el viernes queda
en 0. `Es_Fin_de_Semana` y `Dia_Semana` se dejan ambas disponibles como predictoras porque aportan
información distinta: una captura el patrón binario fin de semana / entre semana, la otra el
detalle día por día (incluyendo el viernes).


### 5.2 Variables predictoras — históricas (lag / rolling)

**Por qué no se puede usar `Num_Accidentes` como predictor:** la variable objetivo `Alto_Riesgo`
se construye directamente a partir de `Num_Accidentes` del mismo día. Si `Num_Accidentes` entra
como predictor, el modelo estaría leyendo la respuesta en vez de predecirla (*data leakage*).

**La solución no es eliminar el historial de accidentes, sino usar solo el de días *anteriores*.**
Que una combinación Localidad-Franja haya tenido muchos accidentes en los últimos 7 o 30 días
es información legítima y disponible en el momento de predecir — es distinto a usar el conteo
del mismo día que se quiere predecir.

Se construyen, por combinación Localidad-Franja y ordenando por fecha:

- `Accidentes_Prom_7d`: promedio móvil de los 7 días **anteriores** (no incluye el día actual).
- `Accidentes_Prom_30d`: promedio móvil de los 30 días **anteriores**.
- `Accidentes_Semana_Anterior`: el conteo exacto de esa misma combinación, 7 días atrás.


In [49]:
dataset = dataset.sort_values(['Localidad', 'Franja_Horaria', 'Fecha_Acc']).reset_index(drop=True)

grupo = dataset.groupby(['Localidad', 'Franja_Horaria'])['Num_Accidentes']

# shift(1) antes del rolling asegura que el día actual NUNCA se incluya en su propio promedio
dataset['Accidentes_Prom_7d'] = (
    grupo.transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
)
dataset['Accidentes_Prom_30d'] = (
    grupo.transform(lambda x: x.shift(1).rolling(window=30, min_periods=1).mean())
)
dataset['Accidentes_Semana_Anterior'] = grupo.transform(lambda x: x.shift(7))

# Flags explícitos: distinguen "sin historial suficiente todavía" (NaN) de
# "el historial existente indica 0 accidentes" (0 real). Se conservan como columnas
# separadas en vez de perder esa distinción al imputar.
cols_historicas = ['Accidentes_Prom_7d', 'Accidentes_Prom_30d', 'Accidentes_Semana_Anterior']
for col in cols_historicas:
    dataset[f'Sin_Historial_{col}'] = dataset[col].isnull().astype(int)

print('Filas sin historial suficiente (primeros 7/30 días de cada combinación):')
print(dataset[[f'Sin_Historial_{c}' for c in cols_historicas]].sum())

dataset[cols_historicas] = dataset[cols_historicas].fillna(0)

dataset[['Localidad', 'Franja_Horaria', 'Fecha_Acc', 'Num_Accidentes'] + cols_historicas].head(10)


Filas sin historial suficiente (primeros 7/30 días de cada combinación):
Sin_Historial_Accidentes_Prom_7d             80
Sin_Historial_Accidentes_Prom_30d            80
Sin_Historial_Accidentes_Semana_Anterior    560
dtype: int64


,Localidad,Franja_Horaria,Fecha_Acc,Num_Accidentes,Accidentes_Prom_7d,Accidentes_Prom_30d,Accidentes_Semana_Anterior
0,ANTONIO NARIÑO,Madrugada,2018-01-01,0,0.000000,0.000000,0.0
1,ANTONIO NARIÑO,Madrugada,2018-01-02,0,0.000000,0.000000,0.0
2,ANTONIO NARIÑO,Madrugada,2018-01-03,0,0.000000,0.000000,0.0
3,ANTONIO NARIÑO,Madrugada,2018-01-04,0,0.000000,0.000000,0.0
4,ANTONIO NARIÑO,Madrugada,2018-01-05,0,0.000000,0.000000,0.0
5,ANTONIO NARIÑO,Madrugada,2018-01-06,0,0.000000,0.000000,0.0
6,ANTONIO NARIÑO,Madrugada,2018-01-07,0,0.000000,0.000000,0.0
7,ANTONIO NARIÑO,Madrugada,2018-01-08,0,0.000000,0.000000,0.0
8,ANTONIO NARIÑO,Madrugada,2018-01-09,1,0.000000,0.000000,0.0
9,ANTONIO NARIÑO,Madrugada,2018-01-10,0,0.142857,0.111111,0.0


### Interpretación

Las variables históricas se calculan con `shift(1)` antes del `rolling`, lo que garantiza que el
promedio de una fila nunca incluya el propio `Num_Accidentes` de esa fila — evitando la fuga de
información.

**Sobre la imputación de los primeros días:** los primeros 7/30 días de cada combinación
Localidad-Franja no tienen historial previo dentro del dataset, así que su promedio queda en
`NaN` antes de imputar. Convertir ese `NaN` en `0` no es exactamente equivalente conceptualmente
— `NaN` significa "no hay información todavía", mientras que `0` significa "el historial
disponible indica cero accidentes". Para no perder esa distinción, se agregan columnas
`Sin_Historial_*` que marcan explícitamente qué filas fueron imputadas. Esto afecta muy pocas
filas dado el horizonte de 7 años del dataset, pero queda documentado y disponible como variable
adicional para el Notebook 4 si se quiere usar.


## 6. Construcción del dataset final

### Variable objetivo binaria: `Alto_Riesgo`

**Corrección importante sobre fuga temporal:** en una primera versión, el umbral de tercil
superior se calculó usando **todo** el período 2018–2024 por combinación Localidad-Franja. Eso
introduce información futura en la definición de "alto riesgo": el criterio para etiquetar un día
de 2018 terminaba incorporando datos de 2019–2024, algo que en el momento real de predicción no
estaría disponible.

**Solución:** se define un punto de corte temporal fijo, `TRAIN_END = 2022-12-31`, y el umbral de
tercil superior se calcula **solo con datos hasta esa fecha** (2018–2022), por combinación
Localidad-Franja. Ese mismo umbral (ya fijo, "congelado") se aplica después a **todas** las filas,
incluyendo el período de prueba (2023–2024) — así el criterio de "alto riesgo" nunca usa
información del futuro respecto al momento en que se definió.

Esta misma partición (`Periodo = train / test`) se conserva como columna en el dataset final, para
que el Notebook 4 reutilice exactamente el mismo split temporal en vez de hacer uno nuevo (aleatorio
o distinto), lo cual rompería la coherencia de esta decisión.


> ⚠️ **Advertencia de fuga de información (data leakage):** `Alto_Riesgo` se calcula a partir
> de `Num_Accidentes` del mismo día. En el Notebook 4, `Num_Accidentes` **no debe incluirse en
> `X`** (variables predictoras) bajo ninguna circunstancia — solo debe usarse para construir `y`.
> Las variables predictoras válidas son las de calendario (`Dia_Semana`, `Mes`,
> `Es_Fin_de_Semana`, `Es_Festivo`) y las históricas (`Accidentes_Prom_7d`, `Accidentes_Prom_30d`,
> `Accidentes_Semana_Anterior`), construidas en la sección 5.


In [50]:
TRAIN_END = '2022-12-31'

dataset['Periodo'] = np.where(dataset['Fecha_Acc'] <= TRAIN_END, 'train', 'test')
print(dataset['Periodo'].value_counts())

# Umbral calculado SOLO con el período de entrenamiento (evita fuga de información temporal)
umbral_train = (
    dataset[dataset['Periodo'] == 'train']
    .groupby(['Localidad', 'Franja_Horaria'])['Num_Accidentes']
    .quantile(2/3)
    .rename('umbral_p66_train')
    .reset_index()
)

dataset = dataset.merge(umbral_train, on=['Localidad', 'Franja_Horaria'], how='left')

# El mismo umbral (fijo, calculado solo con train) se aplica también al período test
dataset['Alto_Riesgo'] = (dataset['Num_Accidentes'] > dataset['umbral_p66_train']).astype(int)

print()
print('Proporción global de Alto_Riesgo:', round(dataset['Alto_Riesgo'].mean(), 4))
print('Proporción en train:', round(dataset[dataset['Periodo']=='train']['Alto_Riesgo'].mean(), 4))
print('Proporción en test:', round(dataset[dataset['Periodo']=='test']['Alto_Riesgo'].mean(), 4))


Periodo
train    146080
test      58480
Name: count, dtype: int64

Proporción global de Alto_Riesgo: 0.1461
Proporción en train: 0.1793
Proporción en test: 0.0631


In [51]:
# Vista rápida: localidades con mayor proporción de combinaciones "alto riesgo" (dentro de train)
dataset[dataset['Periodo']=='train'].groupby('Localidad')['Alto_Riesgo'].mean().sort_values(ascending=False).round(3)


Localidad
CIUDAD BOLÍVAR        0.275
KENNEDY               0.246
ENGATIVÁ              0.242
CHAPINERO             0.221
SUBA                  0.216
TEUSAQUILLO           0.207
BARRIOS UNIDOS        0.205
LOS MÁRTIRES          0.203
USAQUÉN               0.199
FONTIBÓN              0.197
PUENTE ARANDA         0.196
SAN CRISTÓBAL         0.191
SANTA FE              0.178
TUNJUELITO            0.176
RAFAEL URIBE URIBE    0.169
BOSA                  0.153
USME                  0.121
ANTONIO NARIÑO        0.111
CANDELARIA            0.080
SUMAPAZ               0.001
Name: Alto_Riesgo, dtype: float64

### Interpretación

La proporción de observaciones clasificadas como `Alto_Riesgo` es mayor en el período de entrenamiento que en el período de prueba. Esta diferencia evidencia un cambio en la distribución de la variable objetivo entre ambos períodos y deberá ser considerada al interpretar el desempeño de los modelos.

El proyecto ha identificado previamente una reducción en el volumen de registros correspondiente a 2023–2024, la cual podría estar relacionada con cambios en la cobertura o registro de la información. Sin embargo, esta diferencia no permite concluir por sí sola la existencia de subregistro, por lo que esta posibilidad deberá considerarse como una limitación o hipótesis a evaluar.

En consecuencia, el desempeño obtenido sobre 2023–2024 deberá interpretarse teniendo en cuenta que la distribución de la variable objetivo no es igual a la observada durante el período de entrenamiento.

### Validación previa a exportar

Última comprobación de integridad antes de guardar el dataset: estructura, valores nulos y duplicados.


In [52]:
dataset = dataset.drop(columns=['umbral_p66_train'])

dataset.info()


<class 'pandas.DataFrame'>
RangeIndex: 204560 entries, 0 to 204559
Data columns (total 16 columns):
 #   Column                                    Non-Null Count   Dtype        
---  ------                                    --------------   -----        
 0   Localidad                                 204560 non-null  str          
 1   Franja_Horaria                            204560 non-null  str          
 2   Fecha_Acc                                 204560 non-null  datetime64[s]
 3   Num_Accidentes                            204560 non-null  int64        
 4   Dia_Semana                                204560 non-null  str          
 5   Mes                                       204560 non-null  int32        
 6   Es_Fin_de_Semana                          204560 non-null  int64        
 7   Es_Festivo                                204560 non-null  int64        
 8   Accidentes_Prom_7d                        204560 non-null  float64      
 9   Accidentes_Prom_30d                  

In [53]:
dataset.head()


,Localidad,Franja_Horaria,Fecha_Acc,Num_Accidentes,Dia_Semana,Mes,Es_Fin_de_Semana,Es_Festivo,Accidentes_Prom_7d,Accidentes_Prom_30d,Accidentes_Semana_Anterior,Sin_Historial_Accidentes_Prom_7d,Sin_Historial_Accidentes_Prom_30d,Sin_Historial_Accidentes_Semana_Anterior,Periodo,Alto_Riesgo
0,ANTONIO NARIÑO,Madrugada,2018-01-01,0,Monday,1,0,1,0.0,0.0,0.0,1,1,1,train,0
1,ANTONIO NARIÑO,Madrugada,2018-01-02,0,Tuesday,1,0,0,0.0,0.0,0.0,0,0,1,train,0
2,ANTONIO NARIÑO,Madrugada,2018-01-03,0,Wednesday,1,0,0,0.0,0.0,0.0,0,0,1,train,0
3,ANTONIO NARIÑO,Madrugada,2018-01-04,0,Thursday,1,0,0,0.0,0.0,0.0,0,0,1,train,0
4,ANTONIO NARIÑO,Madrugada,2018-01-05,0,Friday,1,0,0,0.0,0.0,0.0,0,0,1,train,0


In [54]:
print('Nulos por columna:')
print(dataset.isnull().sum())
print()
print('Filas duplicadas:', dataset.duplicated().sum())


Nulos por columna:
Localidad                                   0
Franja_Horaria                              0
Fecha_Acc                                   0
Num_Accidentes                              0
Dia_Semana                                  0
Mes                                         0
Es_Fin_de_Semana                            0
Es_Festivo                                  0
Accidentes_Prom_7d                          0
Accidentes_Prom_30d                         0
Accidentes_Semana_Anterior                  0
Sin_Historial_Accidentes_Prom_7d            0
Sin_Historial_Accidentes_Prom_30d           0
Sin_Historial_Accidentes_Semana_Anterior    0
Periodo                                     0
Alto_Riesgo                                 0
dtype: int64

Filas duplicadas: 0


### Interpretación

El dataset final no presenta valores nulos ni filas duplicadas, y los tipos de datos son
consistentes con lo esperado (`Localidad` y `Franja_Horaria` como texto, `Fecha_Acc` como fecha,
`Num_Accidentes` y `Alto_Riesgo` como enteros). Queda listo para exportar.


In [55]:
clave = ['Localidad', 'Franja_Horaria', 'Fecha_Acc']

duplicados_clave = dataset.duplicated(subset=clave).sum()

print(f'Duplicados en Localidad × Franja_Horaria × Fecha_Acc: {duplicados_clave}')

assert duplicados_clave == 0, "Existen duplicados en la granularidad esperada."

Duplicados en Localidad × Franja_Horaria × Fecha_Acc: 0


In [56]:
print("Localidades:", dataset['Localidad'].nunique())
print("Franjas:", dataset['Franja_Horaria'].nunique())
print("Fechas:", dataset['Fecha_Acc'].nunique())

Localidades: 20
Franjas: 4
Fechas: 2557


In [57]:
assert dataset['Localidad'].nunique() == 20
assert dataset['Franja_Horaria'].nunique() == 4
assert dataset['Fecha_Acc'].nunique() == 2557

## 7. Exportación

Se guarda en formato `.parquet` (más eficiente que `.csv` para este volumen y preserva tipos de datos).


In [58]:
dataset["Fecha_Acc"] = dataset["Fecha_Acc"].astype(str)

In [59]:
output_file = PROCESSED_PATH / 'dataset_localidad_franja_fecha.parquet'
dataset.to_parquet(output_file, index=False)

print(f'Guardado: {output_file}')
print(f'Filas: {len(dataset)} | Columnas: {list(dataset.columns)}')


Guardado: ../data/processed/dataset_localidad_franja_fecha.parquet
Filas: 204560 | Columnas: ['Localidad', 'Franja_Horaria', 'Fecha_Acc', 'Num_Accidentes', 'Dia_Semana', 'Mes', 'Es_Fin_de_Semana', 'Es_Festivo', 'Accidentes_Prom_7d', 'Accidentes_Prom_30d', 'Accidentes_Semana_Anterior', 'Sin_Historial_Accidentes_Prom_7d', 'Sin_Historial_Accidentes_Prom_30d', 'Sin_Historial_Accidentes_Semana_Anterior', 'Periodo', 'Alto_Riesgo']


## Resumen de decisiones tomadas en esta fase

| Decisión | Justificación |
|---|---|
| Filtro 2018–2024 | Delimitación temporal del anteproyecto |
| `fillna("NO")` en columnas `Con_*` | El nulo representa "no aplica", no dato faltante |
| Grid completo (incluye ceros) | Representa explícitamente los días sin accidentes y evita trabajar únicamente con días en los que ocurrió un siniestro |
| Granularidad diaria (no solo Localidad × Franja) | 80 combinaciones eran insuficientes para entrenar un modelo robusto |
| Variables de calendario (`Dia_Semana`, `Mes`, `Es_Fin_de_Semana`, `Es_Festivo`) | Predictoras válidas, conocidas de antemano para cualquier fecha futura |
| Variables históricas con `shift(1)` antes del `rolling` (`Accidentes_Prom_7d`, `Accidentes_Prom_30d`, `Accidentes_Semana_Anterior`) | Aportan historial de riesgo real sin incluir el dato del día que se predice |
| Columnas `Sin_Historial_*` | Distinguen "sin historial suficiente" (imputado) de "historial real en cero" |
| Split temporal fijo `Periodo` (train ≤ 2022-12-31 / test 2023–2024) | Necesario para calcular el umbral de `Alto_Riesgo` sin fuga de información futura |
| Umbral de tercil superior calculado **solo con train**, aplicado también a test | Evita que la definición de "alto riesgo" incorpore información del futuro |
| `Num_Accidentes` se conserva en el dataset pero **se excluye de `X`** en el Notebook 4 | Es la misma información usada para construir `Alto_Riesgo` — incluirla sería *data leakage* |
| `Vehiculos` y `Actor_vial` cargadas pero no integradas | Distinta granularidad (por vehículo / por persona); pendiente evaluar en una segunda iteración |

**Siguiente notebook:** `04_Modelado.ipynb` — usar la columna `Periodo` ya construida aquí para el split train/test (no generar uno nuevo), entrenar Regresión Logística, Random Forest y XGBoost con `X` = variables de calendario + históricas, `y` = `Alto_Riesgo`.
